# TinyLlama Fine-Tuning on claims.jsonl

This notebook demonstrates the complete process of fine-tuning the TinyLlama model on the `claims.jsonl` dataset, including data loading, preprocessing, model training, evaluation, and saving the trained model.

## 1. Import Required Libraries

Import all necessary libraries for data handling, preprocessing, model building, and evaluation.

In [2]:
!pip install 'accelerate>=1.1.0'
!pip install -U transformers[torch] accelerate

zsh:1: no matches found: transformers[torch]


In [3]:
# Install required libraries (uncomment if running for the first time)
# !pip install transformers datasets torch pandas scikit-learn matplotlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
import torch

/Users/sofiiapopeniuk/Documents/ml/project/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load and Explore claims.jsonl Dataset

Load the claims.jsonl dataset, display basic statistics, check for missing values, and visualize class distributions.

In [4]:
# Load the claims.jsonl dataset
claims_df = pd.read_json('../data/claims.jsonl', lines=True)

# Display the first few rows
display(claims_df.head())

# Show basic statistics
print(claims_df.info())
print(claims_df.describe(include='all'))

# Check for missing values
print('Missing values per column:')
print(claims_df.isnull().sum())

# Visualize class distribution if label column exists
if 'label' in claims_df.columns:
    claims_df['label'].value_counts().plot(kind='bar')
    plt.title('Class Distribution')
    plt.xlabel('Label')
    plt.ylabel('Count')
    plt.show()

,id,input,output,meta
0,uk_0001,Українська Центральна Рада (УЦР) — це перший у...,{'claims': ['Українська Центральна Рада (УЦР) ...,{'q_id': 'bff38a17-fa1d-4eb0-8d0c-2876adb68ede...
1,uk_0002,Основні етапи діяльності Української Центральн...,{'claims': ['Українська Центральна Рада прогол...,{'q_id': 'b9deff79-a0c8-4278-8e66-319183629c25...
2,uk_0003,Українська Центральна Рада (УЦР) відіграла клю...,{'claims': ['Українська Центральна Рада (УЦР) ...,{'q_id': '5b6dc5f8-36ce-4e53-82dc-0fc04dafb962...
3,uk_0004,"Основні документи, ухвалені Українською Центра...",{'claims': ['Українська Центральна Рада ухвали...,{'q_id': '171a769f-0d52-40b4-ac88-da6f7cddfc6e...
4,uk_0005,Універсальна Центральна Рада (УЦР) стикалася з...,{'claims': ['Універсальна Центральна Рада (УЦР...,{'q_id': '8671738f-8cf4-4997-903d-f8564b155988...


<class 'pandas.DataFrame'>
RangeIndex: 5835 entries, 0 to 5834
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      5835 non-null   str   
 1   input   5835 non-null   str   
 2   output  5835 non-null   object
 3   meta    5835 non-null   object
dtypes: object(2), str(2)
memory usage: 3.1+ MB
None
             id                                              input  \
count      5835                                               5835   
unique     5835                                               5835   
top     uk_0001  Українська Центральна Рада (УЦР) — це перший у...   
freq          1                                                  1   

                                                   output  \
count                                                5835   
unique                                               5835   
top     {'claims': ['Українська Центральна Рада (УЦР) ...   
freq                                     

## 3. Preprocess the Data

Clean and preprocess the data as needed (e.g., handle missing values, encode categorical variables, normalize or standardize features, text processing if applicable).

In [5]:
# Example preprocessing: drop rows with missing input, fill missing output, and basic text cleaning
claims_df = claims_df.dropna(subset=['input'])
if 'output' in claims_df.columns:
    claims_df['output'] = claims_df['output'].fillna('unknown')

# Basic text cleaning (customize as needed)
def clean_text(text):
    return text.strip()

claims_df['input'] = claims_df['input'].apply(clean_text)

# Display cleaned data
display(claims_df.head())

,id,input,output,meta
0,uk_0001,Українська Центральна Рада (УЦР) — це перший у...,{'claims': ['Українська Центральна Рада (УЦР) ...,{'q_id': 'bff38a17-fa1d-4eb0-8d0c-2876adb68ede...
1,uk_0002,Основні етапи діяльності Української Центральн...,{'claims': ['Українська Центральна Рада прогол...,{'q_id': 'b9deff79-a0c8-4278-8e66-319183629c25...
2,uk_0003,Українська Центральна Рада (УЦР) відіграла клю...,{'claims': ['Українська Центральна Рада (УЦР) ...,{'q_id': '5b6dc5f8-36ce-4e53-82dc-0fc04dafb962...
3,uk_0004,"Основні документи, ухвалені Українською Центра...",{'claims': ['Українська Центральна Рада ухвали...,{'q_id': '171a769f-0d52-40b4-ac88-da6f7cddfc6e...
4,uk_0005,Універсальна Центральна Рада (УЦР) стикалася з...,{'claims': ['Універсальна Центральна Рада (УЦР...,{'q_id': '8671738f-8cf4-4997-903d-f8564b155988...


## 4. Split Data into Training and Validation Sets

Split the dataset into training and validation sets using scikit-learn's train_test_split function.

In [6]:
# Split the data (adjust 'label' if your task is supervised)
# Use only 1000 for training and 100 for validation
claims_sample = claims_df.sample(n=1100, random_state=42) if len(claims_df) > 1100 else claims_df.copy()
train_df = claims_sample.iloc[:1000].reset_index(drop=True)
val_df = claims_sample.iloc[1000:1100].reset_index(drop=True)

print(f"Training samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")

Training samples: 1000
Validation samples: 100


## 5. Feature Engineering

Create or select relevant features for model training. For language models, this usually means preparing the text for tokenization.

In [7]:
# Load TinyLlama tokenizer
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Ensure pad_token is set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Prepare datasets for Hugging Face Trainer
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

# Tokenization function for input-output pairs
# We will concatenate input and output claims as text for supervised fine-tuning

def format_example(input_text, output_dict):
    # Convert output dict to string (claims list joined by newlines)
    if isinstance(output_dict, dict):
        output_claims = "\n".join(output_dict.get("claims", []))
    else:
        output_claims = ""
    return f"Input: {input_text}\nClaims:\n{output_claims}"

def tokenize_function(examples):
    texts = [format_example(inp, out) for inp, out in zip(examples["input"], examples["output"])]
    tokens = tokenizer(texts, truncation=True, padding="max_length", max_length=64)  # Reduced max_length
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

Map: 100%|██████████| 100/100 [00:00<00:00, 3181.79 examples/s]


## 6. Build and Compile the Model

Define and load the TinyLlama model for causal language modeling.

In [8]:
# Load TinyLlama model for causal language modeling
model = AutoModelForCausalLM.from_pretrained(model_name)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 4007.72it/s]


## 7. Train the Model

Set up training arguments and train the model using Hugging Face Trainer.

In [9]:
# Set up training arguments
training_args = TrainingArguments(
    output_dir="./tinyllama-claims-finetuned",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=1,  # Reduced batch size
    per_device_eval_batch_size=1,   # Reduced batch size
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=2,
    logging_dir="./logs",
    logging_steps=50,
    fp16=True  # Enable mixed precision if supported
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

# Train the model
trainer.train()

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
/Users/sofiiapopeniuk/Documents/ml/project/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,0.000000,nan
2,0.000000,nan
3,0.000000,nan


Writing model shards: 100%|██████████| 1/1 [00:05<00:00,  5.85s/it]
/Users/sofiiapopeniuk/Documents/ml/project/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:04<00:00,  4.94s/it]
/Users/sofiiapopeniuk/Documents/ml/project/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:04<00:00,  4.65s/it]
/Users/sofiiapopeniuk/Documents/ml/project/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=3000, training_loss=0.018653977711995444, metrics={'train_runtime': 2102.6657, 'train_samples_per_second': 1.427, 'train_steps_per_second': 1.427, 'total_flos': 1191758266368000.0, 'train_loss': 0.018653977711995444, 'epoch': 3.0})

## 8. Evaluate Model Performance

Evaluate the trained model using appropriate metrics on the validation set.

In [11]:
# Evaluate the model on the validation set
results = trainer.predict(val_dataset)
print(results.metrics)

{'test_loss': nan, 'test_runtime': 15.3006, 'test_samples_per_second': 6.536, 'test_steps_per_second': 6.536}


## 9. Save the Trained Model

Save the fine-tuned TinyLlama model and tokenizer for future inference or deployment.

In [12]:
# Save the model and tokenizer
model.save_pretrained("./tinyllama-claims-finetuned")
tokenizer.save_pretrained("./tinyllama-claims-finetuned")
print("Model and tokenizer saved.")

Writing model shards: 100%|██████████| 1/1 [00:04<00:00,  4.89s/it]

Model and tokenizer saved.
